In [35]:
# Right now, your attention is bidirectional — token 2 can see tokens 0, 1, and itself. In a GPT, we're predicting the next token, so token i must only see tokens 0 through i-1. Otherwise, the model "cheats" by looking at the answer.
# Before applying softmax to the attention scores S, we mask out the "future" positions by setting them to -∞. When softmax sees -∞, it outputs exactly 0 for those positions.

import numpy as np


batch_size = 2
num_heads = 2
seq_len = 3
d_model = 4

S = np.random.randn(batch_size, num_heads, seq_len, seq_len)
print(f"Before S {S}")

mask = np.tril(np.ones((seq_len, seq_len)))  # 1s on/below diagonal, 0s above
log_mask = np.where(mask == 1, 0.0, -np.inf)
print(f"\n\nlog mask {log_mask}")

S = S + log_mask
print(f"\n\nAfter S {S}")

## Applying softmax

S_max_sub = S - np.max(S, axis=-1, keepdims=True)
S_max_sub_exp = np.exp(S_max_sub)
S_max_sub_exp_sum = np.sum(S_max_sub_exp, axis=-1, keepdims=True)
S_norm = S_max_sub_exp/S_max_sub_exp_sum

print(f"\n\nS Norm is {S_norm}")

Before S [[[[ 0.36266143 -0.40816716  0.3719728 ]
   [ 1.35534743  0.18213991  0.45727597]
   [ 0.13118488 -1.26261023  1.0790862 ]]

  [[ 1.13372551 -0.98187506 -0.19701013]
   [-1.03109848 -2.17380408  0.611351  ]
   [-2.01849392  0.84143787 -0.43467637]]]


 [[[ 0.70948043 -0.45366015  0.9087264 ]
   [-0.93066887  1.26780082 -1.18311271]
   [ 0.60252428  0.96073705 -0.48257466]]

  [[-1.24031368 -0.76309     1.16085751]
   [ 0.09455191  0.9329995  -1.37825992]
   [-0.43201128 -0.1074595   0.20282781]]]]


log mask [[  0. -inf -inf]
 [  0.   0. -inf]
 [  0.   0.   0.]]


After S [[[[ 0.36266143        -inf        -inf]
   [ 1.35534743  0.18213991        -inf]
   [ 0.13118488 -1.26261023  1.0790862 ]]

  [[ 1.13372551        -inf        -inf]
   [-1.03109848 -2.17380408        -inf]
   [-2.01849392  0.84143787 -0.43467637]]]


 [[[ 0.70948043        -inf        -inf]
   [-0.93066887  1.26780082        -inf]
   [ 0.60252428  0.96073705 -0.48257466]]

  [[-1.24031368        -inf        

In [39]:
# Right now, if you feed the model ["The", "dog", "sat"] or ["sat", "dog", "The"], the attention mechanism will produce the exact same output (just permuted). Why? Because matrix multiplication and dot products are entirely blind to the order of the rows. The model has no concept of "first", "second", or "third".
# As the original Transformer paper states, we must inject some information about the relative or absolute position of the tokens in the sequence.

pos = np.arange(seq_len)[:, np.newaxis]
print(pos)

dim = np.arange(d_model)
print(dim)

div_term = np.exp(dim * -(math.log(10000.0) / d_model))
print(div_term)

PE = pos * div_term
print(f"PE before {PE}")

PE[:, ::2] = np.sin(PE[:, ::2])
PE[:, 1::2] = np.cos(PE[:, 1::2])
print(f"PE after {PE}")


[[0]
 [1]
 [2]]
[0 1 2 3]
[1.e+00 1.e-01 1.e-02 1.e-03]
PE before [[0.e+00 0.e+00 0.e+00 0.e+00]
 [1.e+00 1.e-01 1.e-02 1.e-03]
 [2.e+00 2.e-01 2.e-02 2.e-03]]
PE after [[0.         1.         0.         1.        ]
 [0.84147098 0.99500417 0.00999983 0.9999995 ]
 [0.90929743 0.98006658 0.01999867 0.999998  ]]
